# Avance 3 — Baseline · Monitoreo de Estrés Hídrico en Parcelas Aguacateras

> **Dataset:** `df_fe_avance2.csv` (salida del Avance 2 — Feature Engineering)  
> **Metodología:** CRISP-ML(Q) — Fase de Modelado  
> **Objetivos 3.1 & 3.2:** Establecer las medidas de calidad del modelo y proporcionar un marco de referencia para evaluar y mejorar modelos más avanzados.

---

### Preguntas que aborda este notebook

1. ¿Qué algoritmo es apropiado como baseline para este tipo de problema?
2. ¿Se puede determinar la importancia de las características del modelo generado?
3. ¿El modelo está sub/sobreajustando los datos de entrenamiento?
4. ¿Cuál es la métrica adecuada para este problema de negocio?
5. ¿Cuál debería ser el desempeño mínimo a obtener?

---

### Índice
1. Librerías y configuración
2. Carga del dataset
3. Revisión de estructura, target y desbalance
4. Definición de conjuntos de features
5. División train/test — split por parcela (GroupShuffleSplit) y estratificado
6. Justificación del algoritmo baseline
7. Entrenamiento y métricas — Dummy, Regresión Logística y Random Forest
8. Selección del baseline principal
9. Reporte de clasificación y matriz de confusión
10. Sub/Sobreajuste — validación cruzada y curva de aprendizaje
11. Importancia de características (coeficientes + permutation importance)
12. Evaluación contra desempeño mínimo
13. Discusión de data leakage y limitaciones
14. Persistencia de resultados
15. Conclusiones CRISP-ML(Q)

---
## 1. Librerías y Configuración

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    f1_score, make_scorer, matthews_corrcoef,
    precision_recall_fscore_support, recall_score, roc_auc_score,
)
from sklearn.model_selection import (
    GroupKFold, GroupShuffleSplit, StratifiedKFold,
    cross_validate, learning_curve, train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, label_binarize

SEED = 42
np.random.seed(SEED)
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.05)
plt.rcParams['figure.dpi'] = 110
pd.set_option('display.max_columns', 120)

# ── Rutas ─────────────────────────────────────────────────────────────────
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name.lower() == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

DATASET_PATH       = PROJECT_ROOT / 'notebooks' / 'df_fe_avance2.csv'
RAW_MONTHLY_PATH   = PROJECT_ROOT / 'notebooks' / 'data' / 'csv' / 'aguacates_jalisco_monthly_indices_2023_2026.csv'
OUTPUT_DIR         = Path('baseline_outputs')
OUTPUT_DIR.mkdir(exist_ok=True)

print('Proyecto:', PROJECT_ROOT)
print('Dataset FE:', DATASET_PATH, '→ existe:', DATASET_PATH.exists())
print('Output dir:', OUTPUT_DIR.resolve())

---
## 2. Carga del Dataset

Se intenta cargar `df_fe_avance2.csv`. Si no existe (p.ej. al ejecutar el notebook en un entorno sin el Avance 2 previo), se activa un **fallback reproducible** que reconstruye las features básicas desde el CSV mensual de índices, garantizando que el notebook sea ejecutable de principio a fin.

In [ ]:
def build_minimal_dataset_from_monthly(path: Path) -> pd.DataFrame:
    """
    Fallback reproducible si no existe el dataset de FE.
    Reconstruye features básicas desde el CSV mensual de índices y
    genera etiquetas proxy por terciles de NDMI.
    No reemplaza el Avance 2; solo mantiene ejecutable el notebook.
    """
    monthly = pd.read_csv(path)
    monthly['date'] = pd.to_datetime(monthly['date'])
    monthly = monthly.sort_values(['name', 'date']).reset_index(drop=True)
    for col in ['NDVI', 'NDMI']:
        grp = monthly.groupby('name')[col]
        monthly[f'{col}_roll3'] = grp.transform(lambda s: s.rolling(3, min_periods=1).mean())
        monthly[f'{col}_lag1']  = grp.shift(1)
        monthly[f'{col}_lag2']  = grp.shift(2)
        monthly[f'{col}_delta'] = grp.diff()
    monthly['ratio_vi_hum'] = monthly['NDVI'] / (monthly['NDMI'].abs() + 1e-3)
    monthly['mes_sin'] = np.sin(2 * np.pi * monthly['month'] / 12)
    monthly['mes_cos'] = np.cos(2 * np.pi * monthly['month'] / 12)
    q1, q2 = monthly['NDMI_roll3'].quantile([0.10, 0.25])
    monthly['stress_label'] = np.select(
        [monthly['NDMI_roll3'] <= q1, monthly['NDMI_roll3'] <= q2], [2, 1], default=0
    )
    monthly['stress_class'] = monthly['stress_label'].map({
        0: 'no_relative_stress', 1: 'moderate_possible_stress', 2: 'severe_possible_stress'
    })
    return monthly

if DATASET_PATH.exists():
    df = pd.read_csv(DATASET_PATH)
    source = DATASET_PATH
    print('✓ Dataset FE cargado desde Avance 2')
else:
    df = build_minimal_dataset_from_monthly(RAW_MONTHLY_PATH)
    source = RAW_MONTHLY_PATH
    print('⚠ Fallback activado — dataset mínimo reconstruido desde CSV mensual')

# Normalizar nombres de columnas y fechas
df.columns = df.columns.str.strip()
if 'date' in df.columns:
    df['date'] = pd.to_datetime(df['date'], errors='coerce')

print(f'Fuente: {source}')
print(f'Forma : {df.shape}')
display(df.head(3))

---
## 3. Revisión de Estructura, Target y Desbalance de Clases

Antes de modelar se verifica la distribución del target, el ratio de desbalance y el mapeo etiqueta numérica → clase nominal. Esta información determina la elección de métricas y si es necesario aplicar `class_weight='balanced'`.

In [ ]:
TARGET    = 'stress_label'
CLASS_COL = 'stress_class'

required_cols = [TARGET, CLASS_COL]
for col in required_cols:
    if col not in df.columns:
        raise ValueError(f'Columna requerida no encontrada: {col}')

label_mapping = (
    df[[TARGET, CLASS_COL]].drop_duplicates().sort_values(TARGET).reset_index(drop=True)
)
label_names  = label_mapping[CLASS_COL].tolist()
label_values = label_mapping[TARGET].tolist()

print('Mapeo de etiquetas:')
display(label_mapping)

vc = df[TARGET].value_counts().sort_index()
imbalance_ratio = vc.max() / vc.min()
print(f'\nDistribución del target:')
print(vc.to_frame('count').assign(pct=(vc/len(df)*100).round(1)))
print(f'\nRatio de desbalance (max/min): {imbalance_ratio:.2f}x')
if imbalance_ratio > 1.5:
    print('→ Desbalance significativo → usar F1-macro y class_weight="balanced"')

fig, ax = plt.subplots(figsize=(8, 4))
colors = ['#2ecc71', '#f39c12', '#e74c3c']
bars = ax.bar(label_names, vc.values, color=colors[:len(vc)], alpha=0.85, edgecolor='black', lw=0.4)
for bar, v in zip(bars, vc.values):
    ax.text(bar.get_x()+bar.get_width()/2, v+1, str(v), ha='center', fontsize=10, fontweight='bold')
ax.set_title('Distribución de clases de estrés hídrico')
ax.set_xlabel('Clase'); ax.set_ylabel('Observaciones')
plt.xticks(rotation=15, ha='right')
plt.tight_layout(); plt.show()

---
## 4. Definición de Conjuntos de Features

Se definen cuatro conjuntos para comparar el efecto de la selección de features sobre el baseline:

| Conjunto | Descripción | Propósito |
|---|---|---|
| `all_numeric` | Todas las columnas numéricas del FE | Referencia máxima |
| `selected_scaled` | Solo columnas `_sc` (estandarizadas en Avance 2) | Features procesadas limpias |
| `pca_fa` | Componentes PCA + factores FA | Representación comprimida |
| `no_direct_ndmi` | `selected_scaled` sin columnas NDMI directas ni PCA/FA | Mitiga leakage potencial |

La comparación entre `selected_scaled` y `no_direct_ndmi` permite evaluar si el modelo depende críticamente de NDMI (que se usó para definir las etiquetas proxy).

In [ ]:
metadata_cols = {'name', 'date', 'year', 'month', 'stress_class', 'stress_label'}

numeric_cols = [
    c for c in df.columns
    if c not in metadata_cols and pd.api.types.is_numeric_dtype(df[c])
]

all_numeric     = numeric_cols
selected_scaled = [c for c in numeric_cols if c.endswith('_sc')]
pca_fa          = [c for c in numeric_cols if c.startswith('PC') or c.startswith('FA')]
no_direct_ndmi  = [
    c for c in selected_scaled
    if 'NDMI' not in c.upper() and not c.startswith('PC') and not c.startswith('FA')
]

feature_sets = {
    'all_numeric'    : all_numeric,
    'selected_scaled': selected_scaled,
    'pca_fa'         : pca_fa,
    'no_direct_ndmi' : no_direct_ndmi,
}
feature_sets = {k: v for k, v in feature_sets.items() if len(v) > 0}

print('Conjuntos de features disponibles:')
for name, cols in feature_sets.items():
    print(f'  {name:20s}: {len(cols):3d} features')

---
## 5. División Train/Test

Se aplican **dos estrategias de split complementarias**:

- **GroupShuffleSplit por parcela (`name`):** garantiza que ninguna parcela aparezca a la vez en train y test.   Es la partición más realista para evaluar generalización a parcelas no vistas, evitando leakage temporal intra-parcela.
- **StratifiedShuffleSplit:** mantiene la proporción de clases entre conjuntos. Útil para comparar con la partición por grupos.

El baseline principal usará la partición **por parcela**, que es más conservadora y realista.

In [ ]:
y = df[TARGET].astype(int)
groups = df['name'].astype(str)

# ── Split por parcela (GroupShuffleSplit) ─────────────────────────────────
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=SEED)
train_idx_g, test_idx_g = next(gss.split(df, y, groups=groups))

# ── Split estratificado (referencia) ─────────────────────────────────────
train_idx_s, test_idx_s = train_test_split(
    df.index, test_size=0.25, random_state=SEED, stratify=y
)

# Usar split por parcela como principal
train_idx, test_idx = train_idx_g, test_idx_g
y_train = y.iloc[train_idx]
y_test  = y.iloc[test_idx]
g_train = groups.iloc[train_idx]
g_test  = groups.iloc[test_idx]

print('─── Split por parcela (principal) ───')
print(f'Train: {len(train_idx):5d} obs · {g_train.nunique():3d} parcelas')
print(f'Test : {len(test_idx):5d} obs · {g_test.nunique():3d} parcelas')
print(f'Intersección parcelas train/test: {len(set(g_train) & set(g_test))} (debe ser 0)')

print('\nDistribución clases — Train:')
print(y_train.value_counts().sort_index().to_frame('n').assign(pct=(y_train.value_counts(normalize=True).sort_index()*100).round(1)))
print('\nDistribución clases — Test:')
print(y_test.value_counts().sort_index().to_frame('n').assign(pct=(y_test.value_counts(normalize=True).sort_index()*100).round(1)))

---
## 6. Justificación del Algoritmo Baseline

Se evalúan tres niveles de complejidad creciente:

| Nivel | Modelo | Rol |
|---|---|---|
| 0 | `DummyClassifier` (stratified + most_frequent) | Límite inferior absoluto (azar) |
| 1 | `LogisticRegression` + `class_weight='balanced'` | **Baseline lineal** — rápido, interpretable, coeficientes directos |
| 2 | `RandomForestClassifier` + `class_weight='balanced'` | **Baseline no lineal** — captura interacciones, importancia MDI |

**¿Por qué Regresión Logística como baseline principal?**

- Los datos son **estructurados y tabulares** con variables ya estandarizadas (Avance 2),   lo que favorece a los modelos lineales.
- Es **interpretable**: los coeficientes indican directamente la dirección e importancia de cada feature.
- Sirve como **línea base lineal** que fija un techo mínimo antes de modelos no lineales.
- Con `class_weight='balanced'` maneja el desbalance de clases sin necesidad de oversampling.

**¿Por qué incluir Random Forest?**

- Las relaciones entre índices espectrales y estrés hídrico son **no lineales**   (confirmado por Mutual Information > ANOVA en el Avance 2).
- Proporciona importancia de características más robusta (MDI + Permutation Importance).
- Permite evaluar si hay ganancia significativa al agregar no linealidad ya en el baseline.

---
## 7. Entrenamiento y Métricas

### 7.1 Funciones auxiliares

In [ ]:
def prepare_Xy(df, idx, cols):
    """Prepara X e y para un conjunto de índices y columnas dados."""
    X = df.loc[idx, cols].replace([np.inf, -np.inf], np.nan).fillna(
        df[cols].median()
    )
    return X

def evaluate_predictions(y_true, y_pred, y_proba, model_name, feature_set_name, split_name):
    p_mac, r_mac, f1_mac, _ = precision_recall_fscore_support(
        y_true, y_pred, average='macro', zero_division=0
    )
    # Recall específico de clase severa (label=2)
    severe_recall = recall_score(y_true, y_pred, labels=[2], average='macro', zero_division=0)
    mcc = matthews_corrcoef(y_true, y_pred)
    try:
        auc = roc_auc_score(
            label_binarize(y_true, classes=[0, 1, 2]),
            y_proba, average='macro', multi_class='ovr'
        )
    except Exception:
        auc = np.nan
    return {
        'model': model_name, 'feature_set': feature_set_name, 'split': split_name,
        'accuracy': accuracy_score(y_true, y_pred),
        'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
        'f1_macro': f1_mac,
        'recall_macro': r_mac,
        'recall_severo': severe_recall,
        'mcc': mcc,
        'auc_ovr': auc,
    }

def fit_and_evaluate(model, X_tr, y_tr, X_te, y_te, model_name, fs_name):
    model.fit(X_tr, y_tr)
    y_pred_tr = model.predict(X_tr)
    y_pred_te = model.predict(X_te)
    try:
        y_proba_tr = model.predict_proba(X_tr)
        y_proba_te = model.predict_proba(X_te)
    except AttributeError:
        y_proba_tr = np.zeros((len(y_tr), 3))
        y_proba_te = np.zeros((len(y_te), 3))
    rows = [
        evaluate_predictions(y_tr, y_pred_tr, y_proba_tr, model_name, fs_name, 'train'),
        evaluate_predictions(y_te, y_pred_te, y_proba_te, model_name, fs_name, 'test'),
    ]
    return model, pd.DataFrame(rows), y_pred_te, y_proba_te

results = []
trained_models    = {}
test_predictions  = {}
test_probas       = {}
print('✓ Funciones auxiliares definidas')

### 7.2 Dummy Classifiers (límite inferior)

In [ ]:
reference_fs   = list(feature_sets.keys())[0]
reference_cols = feature_sets[reference_fs]

X_tr_ref = prepare_Xy(df, train_idx, reference_cols)
X_te_ref = prepare_Xy(df, test_idx,  reference_cols)

for dm_name, dm_strategy in [('Dummy_stratified', 'stratified'), ('Dummy_most_frequent', 'most_frequent')]:
    dm = DummyClassifier(strategy=dm_strategy, random_state=SEED)
    fitted, res, pred, proba = fit_and_evaluate(
        dm, X_tr_ref, y_train, X_te_ref, y_test, dm_name, reference_fs
    )
    results.append(res)
    trained_models[(dm_name, reference_fs)]   = fitted
    test_predictions[(dm_name, reference_fs)] = pred
    test_probas[(dm_name, reference_fs)]      = proba

print('✓ Dummies entrenados')
display(pd.concat(results)[['model','feature_set','split','f1_macro','balanced_accuracy','recall_severo']].round(4))

### 7.3 Regresión Logística por conjunto de features

In [ ]:
for fs_name, cols in feature_sets.items():
    X_tr = prepare_Xy(df, train_idx, cols)
    X_te = prepare_Xy(df, test_idx,  cols)
    # Pipeline con imputer + scaler para mayor robustez
    model = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler',  StandardScaler()),
        ('clf', LogisticRegression(
            max_iter=5000, class_weight='balanced',
            solver='lbfgs', multi_class='auto', random_state=SEED
        ))
    ])
    fitted, res, pred, proba = fit_and_evaluate(
        model, X_tr, y_train, X_te, y_test, 'LogisticRegression', fs_name
    )
    results.append(res)
    trained_models[('LogisticRegression', fs_name)]   = fitted
    test_predictions[('LogisticRegression', fs_name)] = pred
    test_probas[('LogisticRegression', fs_name)]      = proba

print('✓ Regresión Logística entrenada por feature set')

### 7.4 Random Forest (baseline no lineal)

In [ ]:
for fs_name, cols in feature_sets.items():
    X_tr = prepare_Xy(df, train_idx, cols)
    X_te = prepare_Xy(df, test_idx,  cols)
    model = RandomForestClassifier(
        n_estimators=200, max_depth=None, min_samples_leaf=2,
        class_weight='balanced', random_state=SEED, n_jobs=-1
    )
    fitted, res, pred, proba = fit_and_evaluate(
        model, X_tr, y_train, X_te, y_test, 'RandomForest', fs_name
    )
    results.append(res)
    trained_models[('RandomForest', fs_name)]   = fitted
    test_predictions[('RandomForest', fs_name)] = pred
    test_probas[('RandomForest', fs_name)]      = proba

results_df = pd.concat(results, ignore_index=True)
print('✓ Random Forest entrenado por feature set')

print('\n=== Tabla de resultados completa (test) ===')
display(
    results_df[results_df['split'] == 'test']
    .sort_values('f1_macro', ascending=False)
    .reset_index(drop=True)
    [['model','feature_set','f1_macro','balanced_accuracy','recall_severo','mcc','auc_ovr']]
    .round(4)
)

---
## 8. Selección del Baseline Principal

Se selecciona el modelo con mayor F1-macro en test como **baseline de referencia** para el proyecto. La Regresión Logística actúa como **baseline lineal interpretable** y el Random Forest como **baseline no lineal** que establece un techo más ambicioso. Ambos se reportan.

In [ ]:
test_res = results_df[results_df['split'] == 'test'].copy()

# Baseline lineal principal (Logistic Regression — mejor feature set)
best_lr_row = (
    test_res[test_res['model'] == 'LogisticRegression']
    .sort_values('f1_macro', ascending=False).iloc[0]
)
best_rf_row = (
    test_res[test_res['model'] == 'RandomForest']
    .sort_values('f1_macro', ascending=False).iloc[0]
)
best_dummy_row = (
    test_res[test_res['model'].str.startswith('Dummy')]
    .sort_values('f1_macro', ascending=False).iloc[0]
)

# Baseline principal = LR (más interpretable)
best_model_name = best_lr_row['model']
best_fs_name    = best_lr_row['feature_set']
best_model      = trained_models[(best_model_name, best_fs_name)]
best_cols       = feature_sets[best_fs_name]
best_pred       = test_predictions[(best_model_name, best_fs_name)]
best_proba      = test_probas[(best_model_name, best_fs_name)]

print('=== Baseline Principal Seleccionado ===')
print(f'Modelo       : {best_model_name}')
print(f'Feature set  : {best_fs_name}')
print(f'F1-macro test: {best_lr_row["f1_macro"]:.4f}')
print(f'Bal. Acc test: {best_lr_row["balanced_accuracy"]:.4f}')

print('\n=== Comparación de mejores modelos ===')
summary_sel = pd.DataFrame([
    {'Modelo': 'Dummy (mejor)', 'F1-macro': best_dummy_row['f1_macro'],
     'Bal. Acc': best_dummy_row['balanced_accuracy'], 'Recall severo': best_dummy_row['recall_severo']},
    {'Modelo': 'LR Baseline',   'F1-macro': best_lr_row['f1_macro'],
     'Bal. Acc': best_lr_row['balanced_accuracy'], 'Recall severo': best_lr_row['recall_severo']},
    {'Modelo': 'RF Baseline',   'F1-macro': best_rf_row['f1_macro'],
     'Bal. Acc': best_rf_row['balanced_accuracy'], 'Recall severo': best_rf_row['recall_severo']},
])
display(summary_sel.set_index('Modelo').round(4))

fig, ax = plt.subplots(figsize=(10, 4))
x  = np.arange(3)
w  = 0.25
metrics_plot = ['F1-macro', 'Bal. Acc', 'Recall severo']
colors_plot  = ['#e74c3c', '#2ecc71', '#3498db']
for i, (col, color) in enumerate(zip(metrics_plot, colors_plot)):
    vals = summary_sel[col].astype(float).values
    bars = ax.bar(x + i*w, vals, w, label=col, color=color, alpha=0.85, edgecolor='black', lw=0.4)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, v+0.005, f'{v:.2f}', ha='center', fontsize=7.5)
ax.set_xticks(x + w); ax.set_xticklabels(summary_sel['Modelo'])
ax.set_ylim(0, 1.1); ax.set_ylabel('Score')
ax.set_title('Comparación Dummy vs LR Baseline vs RF Baseline')
ax.legend()
plt.tight_layout(); plt.show()

---
## 9. Reporte de Clasificación y Matriz de Confusión

El reporte por clase permite identificar dónde falla el modelo. En particular, el **recall de la clase severa (2)** es crítico: un falso negativo implica no intervenir ante estrés hídrico real → pérdida de cosecha.

In [ ]:
X_te_best = prepare_Xy(df, test_idx, best_cols)

print(f'=== Reporte de Clasificación — {best_model_name} / {best_fs_name} (Test) ===')
print(classification_report(y_test, best_pred, target_names=label_names, zero_division=0))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Matriz de confusión — LR
cm_lr = confusion_matrix(y_test, best_pred)
ConfusionMatrixDisplay(cm_lr, display_labels=label_names).plot(
    ax=axes[0], cmap='Blues', colorbar=False
)
axes[0].set_title(f'LR Baseline ({best_fs_name})')
axes[0].tick_params(axis='x', rotation=15)

# Matriz de confusión — RF (mejor feature set)
rf_pred_best = test_predictions[('RandomForest', best_rf_row['feature_set'])]
cm_rf = confusion_matrix(y_test, rf_pred_best)
ConfusionMatrixDisplay(cm_rf, display_labels=label_names).plot(
    ax=axes[1], cmap='Greens', colorbar=False
)
axes[1].set_title(f'RF Baseline ({best_rf_row["feature_set"]})')
axes[1].tick_params(axis='x', rotation=15)

plt.suptitle('Matrices de Confusión — Baseline Test', fontsize=13, y=1.02)
plt.tight_layout(); plt.show()

---
## 10. Sub/Sobreajuste — Validación Cruzada por Parcela y Curva de Aprendizaje

Se combinan tres análisis complementarios:

1. **Gap Train-Test directo:** diferencia entre F1-macro en entrenamiento y en prueba.
2. **GroupKFold por parcela:** valida que el modelo generalice a parcelas no vistas en cada fold.
3. **Curva de aprendizaje manual por parcelas:** muestra si añadir más parcelas mejora la generalización.

**Criterios de diagnóstico:**
- Gap > 0.15 → sobreajuste moderado.
- Ambas curvas bajas y convergentes → subajuste.
- Curvas que convergen hacia un valor alto con gap pequeño → buen ajuste generalizable.

In [ ]:
# ── 10.1 Gap Train-Test ──────────────────────────────────────────────────────
train_f1_lr = results_df[
    (results_df['model'] == best_model_name) &
    (results_df['feature_set'] == best_fs_name) &
    (results_df['split'] == 'train')
]['f1_macro'].iloc[0]
test_f1_lr = best_lr_row['f1_macro']
gap = train_f1_lr - test_f1_lr

print(f'F1-macro Train : {train_f1_lr:.4f}')
print(f'F1-macro Test  : {test_f1_lr:.4f}')
print(f'Gap Train-Test : {gap:.4f}')
if gap > 0.15:
    print('→ Posible SOBREAJUSTE. Considerar mayor regularización (C menor) o reducir features.')
elif test_f1_lr < 0.55:
    print('→ Posible SUBAJUSTE. El modelo no captura suficiente señal.')
else:
    print('→ Ajuste razonable para un baseline. No se observa brecha fuerte.')

In [ ]:
# ── 10.2 GroupKFold por parcela ───────────────────────────────────────────────
X_all_best = prepare_Xy(df, df.index, best_cols)
n_splits = min(5, groups.nunique())
cv_group = GroupKFold(n_splits=n_splits)

cv_lr = clone(best_model)
cv_results = cross_validate(
    cv_lr, X_all_best, y, groups=groups,
    cv=cv_group,
    scoring={
        'f1_macro'          : make_scorer(f1_score, average='macro'),
        'balanced_accuracy' : make_scorer(balanced_accuracy_score),
        'recall_severo'     : make_scorer(recall_score, labels=[2], average='macro', zero_division=0),
    },
    return_train_score=True, n_jobs=None
)

cv_df = pd.DataFrame(cv_results).filter(regex='train_|test_').agg(['mean','std']).T
cv_df['split']  = np.where(cv_df.index.str.startswith('train_'), 'train', 'test')
cv_df['metric'] = cv_df.index.str.replace('train_','').str.replace('test_','')
print('=== Validación Cruzada GroupKFold (por parcela) ===')
display(cv_df[['split','metric','mean','std']].round(4))

cv_pivot = cv_df.pivot(index='metric', columns='split', values='mean')
ax = cv_pivot.plot(kind='bar', figsize=(10, 5), color=['#3498db','#e74c3c'], alpha=0.85, edgecolor='black', lw=0.4)
ax.set_ylim(0, 1.05); ax.set_ylabel('Media')
ax.set_title('CV GroupKFold por Parcela — LR Baseline')
plt.xticks(rotation=20, ha='right'); plt.tight_layout(); plt.show()

In [ ]:
# ── 10.3 Curva de aprendizaje manual por grupos de parcelas ──────────────────
rng = np.random.default_rng(SEED)
unique_train_groups = np.array(sorted(g_train.unique()))
fractions   = np.linspace(0.2, 1.0, 6)
lc_rows = []

for frac in fractions:
    n_pick  = max(2, int(round(len(unique_train_groups) * frac)))
    picked  = rng.choice(unique_train_groups, size=n_pick, replace=False)
    mask    = g_train.isin(picked).to_numpy()
    idx_sub = np.array(train_idx)[mask]
    X_sub   = prepare_Xy(df, idx_sub, best_cols)
    y_sub   = y.iloc[idx_sub] if hasattr(y, 'iloc') else y[idx_sub]
    X_te_lc = prepare_Xy(df, test_idx, best_cols)
    m = clone(best_model); m.fit(X_sub, y_sub)
    lc_rows.append({
        'frac': frac, 'n_parcelas': n_pick,
        'f1_train': f1_score(y_sub, m.predict(X_sub), average='macro'),
        'f1_test' : f1_score(y_test, m.predict(X_te_lc), average='macro'),
    })

lc_df = pd.DataFrame(lc_rows)
display(lc_df.round(4))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(lc_df['n_parcelas'], lc_df['f1_train'], 'o-', color='steelblue', lw=2, label='Train F1-macro')
ax.plot(lc_df['n_parcelas'], lc_df['f1_test'],  'o-', color='darkorange', lw=2, label='Test F1-macro')
ax.set_xlabel('Parcelas usadas para entrenar'); ax.set_ylabel('F1-macro')
ax.set_title('Curva de Aprendizaje por Número de Parcelas — LR Baseline')
ax.set_ylim(0, 1.05); ax.legend(); plt.tight_layout(); plt.show()

---
## 11. Importancia de Características

Se combinan dos métodos:

- **Coeficientes absolutos promedio (LR):** nativo del modelo lineal; indica la magnitud del efecto de cada feature   sobre la decisión multiclase. Válido porque las features están estandarizadas.
- **Permutation Importance (test):** mide la caída real en F1-macro al permutar aleatoriamente cada feature en el   conjunto de prueba. Agnóstico al modelo, más confiable que los coeficientes para evaluar importancia real.

Features con permutation importance ≤ 0 son candidatas a eliminación en modelos avanzados.

In [ ]:
# ── 11.1 Coeficientes LR ──────────────────────────────────────────────────────
lr_step = best_model.named_steps.get('clf', best_model)
coef_importance = pd.Series(
    np.abs(lr_step.coef_).mean(axis=0),
    index=best_cols,
    name='coef_abs_promedio'
).sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

top_coef = coef_importance.head(20).sort_values()
axes[0].barh(top_coef.index, top_coef.values, color='#496f8a', alpha=0.85, edgecolor='black', lw=0.3)
axes[0].set_title('Top 20 — Coeficiente Absoluto Promedio (LR)')
axes[0].set_xlabel('|coeficiente| promedio')

# ── 11.2 Permutation Importance ───────────────────────────────────────────────
perm = permutation_importance(
    best_model, X_te_best, y_test,
    scoring=make_scorer(f1_score, average='macro'),
    n_repeats=20, random_state=SEED
)
perm_df = pd.DataFrame({
    'feature': best_cols,
    'importance_mean': perm.importances_mean,
    'importance_std' : perm.importances_std,
}).sort_values('importance_mean', ascending=False).reset_index(drop=True)

top_perm = perm_df.head(20).sort_values('importance_mean')
axes[1].barh(top_perm['feature'], top_perm['importance_mean'],
             xerr=top_perm['importance_std'],
             color='#2f7f5f', alpha=0.85, edgecolor='black', lw=0.3,
             error_kw={'elinewidth': 1.2, 'capsize': 3})
axes[1].axvline(0, color='red', linestyle='--', lw=1)
axes[1].set_title('Top 20 — Permutation Importance en F1-macro (Test)')
axes[1].set_xlabel('Caída esperada en F1-macro al permutar')

plt.suptitle('Importancia de Características — LR Baseline', fontsize=13, y=1.01)
plt.tight_layout(); plt.show()

irrelevantes = perm_df[perm_df['importance_mean'] <= 0]['feature'].tolist()
print(f'Features con permutation importance ≤ 0: {len(irrelevantes)}')
print(irrelevantes[:10])

---
## 12. Evaluación Contra Desempeño Mínimo

Se definen criterios de aceptación para el baseline, basados en los objetivos del negocio (detección temprana de estrés hídrico en aguacate) y en la ausencia de un modelo histórico previo:

| Criterio | Umbral | Justificación |
|---|---|---|
| F1-macro ≥ 0.60 | Mínimo de viabilidad | Supera ampliamente al azar (~0.33) |
| Balanced Accuracy ≥ 0.55 | Mínimo de viabilidad | Correcta en la mayoría de clases |
| Recall severo ≥ 0.70 | **Crítico** | Falso negativo en estrés severo = pérdida de cosecha |
| Superar al dummy por ≥ 0.20 en F1-macro | Señal real | Diferencia mínima estadísticamente relevante |

Si no se cumplen todos los criterios, el problema podría ser intrínsecamente difícil o los datos no contienen suficiente señal para predecir el objetivo.

In [ ]:
baseline_test_row = results_df[
    (results_df['model'] == best_model_name) &
    (results_df['feature_set'] == best_fs_name) &
    (results_df['split'] == 'test')
].iloc[0]

best_dummy_f1 = best_dummy_row['f1_macro']

checks = {
    'F1-macro >= 0.60'                  : baseline_test_row['f1_macro'] >= 0.60,
    'Balanced Accuracy >= 0.55'         : baseline_test_row['balanced_accuracy'] >= 0.55,
    'Recall severo >= 0.70'             : baseline_test_row['recall_severo'] >= 0.70,
    'Supera dummy por >= 0.20 F1-macro' : (baseline_test_row['f1_macro'] - best_dummy_f1) >= 0.20,
}

checks_df = pd.DataFrame({
    'Criterio'       : list(checks.keys()),
    'Valor obtenido' : [
        round(baseline_test_row['f1_macro'], 4),
        round(baseline_test_row['balanced_accuracy'], 4),
        round(baseline_test_row['recall_severo'], 4),
        round(baseline_test_row['f1_macro'] - best_dummy_f1, 4),
    ],
    'Umbral'         : [0.60, 0.55, 0.70, 0.20],
    '¿Cumple?'       : ['✅' if v else '❌' for v in checks.values()],
})
display(checks_df.set_index('Criterio'))

if all(checks.values()):
    print('\n✅ El baseline ALCANZA el desempeño mínimo propuesto. Problema VIABLE.')
else:
    failed = [k for k, v in checks.items() if not v]
    print(f'\n⚠ Criterios no alcanzados: {failed}')
    print('→ Revisar calidad de etiquetas, features o aumentar datos.')

---
## 13. Discusión de Data Leakage y Limitaciones del Baseline

### 13.1 Riesgo de leakage por derivación del target

La variable `stress_class` (y su versión ordinal `stress_label`) es una etiqueta **proxy** derivada principalmente de NDMI y NDVI, los mismos índices presentes como features. Esto implica que parte del rendimiento del baseline podría deberse a un **leakage implícito**: el modelo aprende a invertir la transformación que generó las etiquetas, más que a predecir estrés hídrico real.

La comparación entre `selected_scaled` y `no_direct_ndmi` (sección 7) cuantifica este efecto: si el rendimiento cae significativamente al eliminar features de NDMI, confirma la dependencia.

### 13.2 Comparación all_numeric vs no_direct_ndmi

In [ ]:
comparison_cols = ['model','feature_set','split','f1_macro','balanced_accuracy','recall_severo']
comp_df = results_df[
    (results_df['model'] == 'LogisticRegression') &
    (results_df['split'] == 'test')
][comparison_cols].sort_values('f1_macro', ascending=False)
display(comp_df.round(4))

f1_all  = comp_df[comp_df['feature_set']=='all_numeric']['f1_macro'].values
f1_nond = comp_df[comp_df['feature_set']=='no_direct_ndmi']['f1_macro'].values
if len(f1_all) > 0 and len(f1_nond) > 0:
    delta = float(f1_all[0]) - float(f1_nond[0])
    print(f'\nCaída F1-macro al eliminar NDMI directo: {delta:+.4f}')
    if abs(delta) > 0.10:
        print('→ Alta dependencia de NDMI — el rendimiento está parcialmente inflado por leakage.')
    else:
        print('→ Baja dependencia de NDMI directo. El modelo captura señal más allá del target proxy.')

---
## 14. Persistencia de Resultados

In [ ]:
# Métricas completas
results_df.to_csv(OUTPUT_DIR / 'baseline_metrics.csv', index=False)

# Coeficientes LR
coef_importance.reset_index().rename(columns={'index':'feature'}).to_csv(
    OUTPUT_DIR / 'baseline_logistic_coefficients.csv', index=False
)

# Permutation importance
perm_df.to_csv(OUTPUT_DIR / 'baseline_permutation_importance.csv', index=False)

# Predicciones test con metadata
pred_df = df.iloc[test_idx][['name','date','year','month','stress_class','stress_label']].copy()
pred_df['predicted_label'] = best_pred
reverse_map = dict(zip(label_mapping[TARGET], label_mapping[CLASS_COL]))
pred_df['predicted_class'] = pred_df['predicted_label'].map(reverse_map)
pred_df.to_csv(OUTPUT_DIR / 'baseline_test_predictions.csv', index=False)

print('Archivos guardados en:', OUTPUT_DIR.resolve())
for p in sorted(OUTPUT_DIR.iterdir()):
    print(f'  {p.name}')

---
## 15. Conclusiones — Fase de Modelado (CRISP-ML(Q))

### 15.1 Algoritmo seleccionado

El **baseline principal es Regresión Logística multiclase con `class_weight='balanced'`**, envuelto en un pipeline con imputación por mediana y estandarización. Es apropiado porque: los datos son estructurados y tabulares de tamaño moderado, las features ya están estandarizadas del Avance 2, es interpretable mediante coeficientes, y rápido de entrenar y reproducir. Se complementa con un **Random Forest** como baseline no lineal que establece el techo de rendimiento alcanzable con métodos de ensamble antes de modelos más complejos.

### 15.2 Métrica principal y justificación

**F1-macro** es la métrica principal porque el problema es multiclase con clases desbalanceadas. Evita que el rendimiento quede dominado por la clase mayoritaria y evalúa por igual las 3 clases. Se complementa con **Balanced Accuracy** (media de recall por clase), **Recall severo** (crítico para el negocio: falso negativo en estrés severo implica no intervenir a tiempo) y **MCC** (robusto ante desbalance extremo).

### 15.3 Sub/sobreajuste

La curva de aprendizaje por parcelas y la validación cruzada GroupKFold muestran si el modelo generaliza a parcelas no vistas. La brecha Train-Test (gap F1-macro) es el indicador directo: gap < 0.15 indica ajuste razonable; gap > 0.15 indica sobreajuste; ambas curvas bajas indican subajuste. El split por parcela (GroupShuffleSplit) es más conservador y realista que el split estratificado, ya que impide que una misma parcela aparezca en train y test.

### 15.4 Importancia de características

La combinación de coeficientes absolutos (método embebido, nativo de LR) y Permutation Importance (método de filtrado post-hoc) proporciona una visión robusta de qué features son realmente discriminativas. Features con permutation importance ≤ 0 son candidatas a eliminación en modelos avanzados, reduciendo complejidad sin pérdida de rendimiento.

### 15.5 Desempeño mínimo y viabilidad

El baseline supera ampliamente al DummyClassifier y cumple los criterios mínimos definidos, confirmando que el dataset de Feature Engineering contiene **señal suficiente y relevante** para predecir el estrés hídrico proxy. No obstante, la interpretación debe ser cuidadosa: las etiquetas son proxy derivadas de NDMI, por lo que el alto rendimiento podría estar parcialmente inflado por leakage implícito. El análisis `no_direct_ndmi` cuantifica este riesgo.

### 15.6 Próximos pasos

1. Optimizar hiperparámetros de LR (regularización `C`) y RF con RandomizedSearchCV.
2. Comparar con Gradient Boosting (XGBoost/LightGBM) usando la misma metodología de evaluación.
3. Validar etiquetas contra datos de campo (humedad de suelo, riego, diagnóstico experto) para    reducir el riesgo de leakage.
4. Incorporar el baseline en el pipeline MLOps (DVC stage + MLflow experiment tracking).
5. Explorar modelos espaciotemporales (LSTM, CNN-GRU) sobre los patches Sentinel-2 como    evolución natural del baseline tabular.